In [1]:
import pandas as pd

In [2]:
import os, pickle

In [3]:
with open("../../../training_data/7.Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pickle.load(f)

len(extras_featuresd), extras_featuresd

(30,
 {'21du':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       21du               5             E          264            R   
  1       21du               5             E          265            R   
  2       21du               5             E          266            R   
  3       21du               5             E          267            R   
  4       21du               5             E          268            R   
  ..       ...             ...           ...          ...          ...   
  273     21du               5             E          546            R   
  274     21du               5             E          547            R   
  275     21du               5             E          548            R   
  276     21du               5             E          549            R   
  277     21du               5             E          550            R   
  
                      

In [4]:
all_extras_seqsd = pd.read_pickle("all_extras_seqsd.pkl")

len(all_extras_seqsd), all_extras_seqsd

(30,
 {('21du',
   '5',
   'E'): 'SNCLKEANEVANQILNLTADGQNLTSANITNIVEQVKRIVNKEENIDITLGSTLMNIFSNILSSSDSDLLESSSEALKTIDELAFKIDLNSTSHVNITTRNLALSVSSLLPGTNAISNFSIGLPSNNESYFQMDFESGQVDPLASVILPPNLLENLSPEDSVLVRRAQFTFFNKTGLFQDVGPQRKTLVSYVMACSIGNITIQNLKDPVQIKIKHTRTQEVHHPICAFWDLNKNKSFGGWNTSGCVAHRDSDASETVCLCNHFTHFGVLMDLPRSASQLDARNTKVLTFISYIGCGISAIFSAATLLTYVAFEKLRRDYPSKILMNLSTALLFLNLLFLLDGWITSFNVDGLCIAVAVLLHFFLLATFTWMGLEAIHMYIALVKVFNTYIRRYILKFCIIGWGLPALVVSVVLASRNNNEVYGKESYGKEKGDEFCWIQDPVIFYVTCAGYFGVMFFLNIAMFIVVMVQICGRNGKRSNRTLREEVLRNLRSVVSLTFLLGMTWGFAFFAWGPLNIPFMYLFSIFNSLQGLFIFIFHCAMKENVQKQWRQHLCCGRFRLEFLEVLFQGPWSHPQFEKGGGSGGGSGGSAWSHPQFEKDYKDDDDK',
  ('22mj',
   '1',
   'A'): 'MSSRKLSGPKGRRLSIHVVTWNVASAAPPLDLSDLLQLNNRNLNLDIYVIGLQELNSGIISLLSDAAFNDSWSSFLMDVLSPLSFIKVSHVRMQGILLLVFAKYQHLPYIQILSTKSTPTGLFGYWGNKGGVNICLKLYGYYVSIINCHLPPHISNNYQRLEHFDRILEMQNCEGRDIPNILDHDLIIWFGDMNFRIEDFGLHFVRESIKNRCYGGLWEKDQLSIAKKHDPLLREFQEGRLLFPPTYKFDRNSNDYDTSEKKRKPAWTDRILWRLKRQPAKANPSGFLLTQKDYSSHMTYGISDHKPVSGTFDLELKPLVSA',
  ('

In [5]:
extras_seqsd = {}

for pid, seq in all_extras_seqsd.items():
    pdb, entid, chainid = pid
    if pdb in extras_featuresd:
        assert pdb not in extras_seqsd, pdb
        extras_seqsd[pdb] = seq

len(extras_seqsd), extras_seqsd

(30,
 {'21du': 'SNCLKEANEVANQILNLTADGQNLTSANITNIVEQVKRIVNKEENIDITLGSTLMNIFSNILSSSDSDLLESSSEALKTIDELAFKIDLNSTSHVNITTRNLALSVSSLLPGTNAISNFSIGLPSNNESYFQMDFESGQVDPLASVILPPNLLENLSPEDSVLVRRAQFTFFNKTGLFQDVGPQRKTLVSYVMACSIGNITIQNLKDPVQIKIKHTRTQEVHHPICAFWDLNKNKSFGGWNTSGCVAHRDSDASETVCLCNHFTHFGVLMDLPRSASQLDARNTKVLTFISYIGCGISAIFSAATLLTYVAFEKLRRDYPSKILMNLSTALLFLNLLFLLDGWITSFNVDGLCIAVAVLLHFFLLATFTWMGLEAIHMYIALVKVFNTYIRRYILKFCIIGWGLPALVVSVVLASRNNNEVYGKESYGKEKGDEFCWIQDPVIFYVTCAGYFGVMFFLNIAMFIVVMVQICGRNGKRSNRTLREEVLRNLRSVVSLTFLLGMTWGFAFFAWGPLNIPFMYLFSIFNSLQGLFIFIFHCAMKENVQKQWRQHLCCGRFRLEFLEVLFQGPWSHPQFEKGGGSGGGSGGSAWSHPQFEKDYKDDDDK',
  '22mj': 'MSSRKLSGPKGRRLSIHVVTWNVASAAPPLDLSDLLQLNNRNLNLDIYVIGLQELNSGIISLLSDAAFNDSWSSFLMDVLSPLSFIKVSHVRMQGILLLVFAKYQHLPYIQILSTKSTPTGLFGYWGNKGGVNICLKLYGYYVSIINCHLPPHISNNYQRLEHFDRILEMQNCEGRDIPNILDHDLIIWFGDMNFRIEDFGLHFVRESIKNRCYGGLWEKDQLSIAKKHDPLLREFQEGRLLFPPTYKFDRNSNDYDTSEKKRKPAWTDRILWRLKRQPAKANPSGFLLTQKDYSSHMTYGISDHKPVSGTFDLELKPLVSA',
  '6s3a': 'GSKCRIEPVCLLLHGSPGAGKSVATNLIG

# Make predictions

deepallo was run in its own environment in /data/fnerin/envs_other_tools, created with the requirements_conda.txt file

changes to inference.py regarding paths and fpocket execution were made

In [6]:
os.system(f"ln -s /home/fnerin/miniconda3/envs/pybiomed/bin/fpocket fpocket") # os.getcwd().rsplit('/', 3)[0] --> absolute path of allodb_new

ln: failed to create symbolic link 'fpocket': File exists


256

In [7]:
from pathlib import Path
from Bio.SeqUtils import seq3

In [8]:
for pdb, feats in extras_featuresd.items():
    if not os.path.isfile(f"{pdb}/{pdb}.pkl"):
        os.makedirs(pdb, exist_ok=True)
    
        # The script is hardcoded for rcsb-downloaded .pdb files; some PDBs do not have .pdb available anymore, so the SEQRES is reconstructed below
        # if not os.path.isfile(f"{pdb}/{pdb}.pdb"):
        #     os.system(f"ln -s {os.getcwd().rsplit('/', 1)[0]}/structures/{pdb}.pdb {pdb}/{pdb}.pdb")
        #     # os.getcwd().rsplit('/', 1)[0] --> absolute path of previous directory
        chains = feats[('Residues', 'auth_asym_id')].unique()
        if len(chains) == 1:
            chain_id = chains.item()

            # SEQRES in RCSB .pdb coincides with the pdbx-deposited sequence (canonical) in RCSB .cif
            seq = extras_seqsd[pdb]
            seqres = [
                f"SEQRES {i//13 + 1:>3} {chain_id} {len(seq):>4}  "
                + " ".join(seq3(aa).upper() for aa in seq[i:i+13])
                + "\n"
                for i in range(0, len(seq), 13)
            ]

            with open(f"{pdb}/{pdb}.pdb", "w") as out, open(f"../structures/{pdb}.pdb") as inp:
                out.writelines(seqres)
                out.writelines(inp)

            
            os.system(f"python deepallo/inference.py {pdb} {chain_id}") # If it is not a single chain, .item() will fail
        else:
            assert False
            # for chain in chains:
            #     if os.path.isdir(f"{pdb}_{chain}"): continue
            #     os.makedirs(pdb, exist_ok=True)
            #     os.system(f"python deepallo/inference.py {pdb} {chain}") # If it is not a single chain, .item() will fail
            #     os.system(f"mv {pdb} {pdb}_{chain}")

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/huggingface_hub/file_download.py:834: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 328 files:  29%|██▊       | 94/328 [00:00<00:00, 923.73it/s]

Fetching 328 files: 100%|██████████| 328/328 [00:00<00:00, 857.64it/s]


***** POCKET HUNTING BEGINS ***** 


***** POCKET HUNTING ENDS ***** 


mv: '21du/21du_out' and '21du/21du_out' are the same file


...

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estima

(0.11640623211860657, ['710', '708', '467', '711', '474', '712', '471'])
(0.05958966165781021, ['706', '318', '311', '315', '209', '192', '473', '188', '597', '196', '187', '193', '708', '195', '698', '319', '317', '212', '184', '702', '308', '191', '593', '470', '325', '208', '189', '474', '333', '320', '594', '326', '705', '186', '701', '466', '331', '321', '332', '205'])
(0.030890583992004395, ['213', '263', '210', '217', '264', '214', '267'])


In [10]:
# Any missing
for pdb, feats in extras_featuresd.items():
    if not os.path.isfile(f"{pdb}/{pdb}.pkl"):
        print("Missing:", pdb)

# Processing

In [11]:
import pandas as pd

In [12]:
results = {
    pdb: {
        pocket: {
            "prob": pocketd["prob"],
            "residues": pd.DataFrame(pocketd["res"])
        }
        for pocket, pocketd in pd.read_pickle(f"{pdb}/{pdb}.pkl").items()
    }
    for pdb in extras_featuresd
    # if pdb != "8aq6"
}

len(results), results

(30,
 {'21du': {0: {'prob': 0.042108431458473206,
    'residues':    auth_asym_id auth_seq_id
    0             R         847
    1             R        1096
    2             R         850
    3             R        1087
    4             R         856
    5             R        1092
    6             R         852
    7             R        1093
    8             R         842
    9             R         844
    10            R        1099
    11            R         849
    12            R         848
    13            R        1084
    14            R         853
    15            R         855},
   1: {'prob': 0.03396913409233093,
    'residues':    auth_asym_id auth_seq_id
    0             R        1035
    1             R         953
    2             R         949
    3             R         950
    4             R        1032
    5             R         977
    6             R        1027
    7             R         945
    8             R        1028
    9             R     

In [13]:
from Bio import PDB

def process_pocket(f):
    s = PDB.PDBParser(QUIET=True).get_structure("protein", f)
    return pd.DataFrame((
        {
            "auth_asym_id": chain.id,
            "auth_seq_id": res.get_id()[1],
            "pdbx_PDB_ins_code": res.get_id()[2] if res.get_id()[2] != " " else "?"
        }
        for model in s
        for chain in model
        for res in chain
    ), dtype=str).drop_duplicates()

In [14]:
processed_results = {}

for pdb in extras_featuresd:
    # if pdb == "8aq6": continue
    processed_results[pdb] = {}
    
    pdbpath = f"{pdb}/{pdb}_out/pockets"
    for pocketf in os.listdir(pdbpath):
        if pocketf.endswith("_atm.pdb"):
            pocketn = pocketf.replace("_atm.pdb", "")
            res = process_pocket(f"{pdbpath}/{pocketf}")

            d = {"pred": 0, "residues": res}
            
            existing = tuple(
                pocket
                for pocket, pocketd in results[pdb].items()
                if len( pocketd["residues"].merge(res[["auth_asym_id", "auth_seq_id"]]) ) == len( pocketd["residues"] )
            )
            if len(existing):
                d.update({"pred": 1, "prob": results[pdb][existing[0]]["prob"]})
            elif len(existing) >= 2:
                raise Exception("More than 1 matching + pred pocket" + pdb + pocketn)

            processed_results[pdb][pocketn] = d
    
    assert sum(v["pred"] for v in processed_results[pdb].values()) == 3, "Not all + pred pockets were picked up" + pdb
    processed_results[pdb] = dict(sorted(
        processed_results[pdb].items(), 
        key = lambda x: (
            x[-1].get("prob", 0), 
            1/int(x[0].replace("pocket", ""))
        ), 
        reverse = True
    ))
    
len(processed_results), processed_results

(30,
 {'21du': {'pocket1': {'pred': 1,
    'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
    0             R        1099                 ?
    1             R        1087                 ?
    2             R        1084                 ?
    3             R         852                 ?
    4             R         855                 ?
    5             R         856                 ?
    6             R         853                 ?
    7             R        1096                 ?
    8             R         842                 ?
    9             R        1093                 ?
    10            R        1092                 ?
    11            R         850                 ?
    12            R         847                 ?
    13            R         844                 ?
    14            R         848                 ?
    15            R         849                 ?,
    'prob': 0.042108431458473206},
   'pocket2': {'pred': 1,
    'residues':    auth_asym_id auth_

In [15]:
resultsf = "deepallo_results.pkl"

with open(resultsf, "wb") as f:
    pickle.dump(processed_results, f)